[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/metaflow-certified/notebooks/day-10-feature-pipelines.ipynb#scrollTo=a1b2c3d4)

---
# Day 10 · Feature Engineering Pipelines
**certified-journeys / metaflow-certified** · Practice · Feature Engineering

> **Goal for today:** Build a complete multi-step Metaflow feature engineering pipeline that loads raw data, cleans nulls, engineers features, stores artifacts, and validates schema invariants.


In [ ]:
%pip install -q metaflow scikit-learn pandas numpy


## Step 1 · The Metaflow Flow Anatomy

A Metaflow flow is a Python class that inherits from `FlowSpec`. Each method decorated with `@step` is a DAG node. Data flows between steps via `self` — Metaflow persists every attribute you set on `self` as a named artifact.

| Component | Purpose |
|---|---|
| `FlowSpec` | Base class — defines the DAG |
| `@step` | Marks a method as a pipeline step |
| `self.next(...)` | Declares edges to subsequent steps |
| `self.X = value` | Stores an artifact (persisted automatically) |
| `run()` | Entry point — called when you execute the script |

Since Metaflow flows must be run as scripts (not interactively), we use `%%writefile` to write the flow to a `.py` file and then run it with `!python`.


In [ ]:
# Verify imports work before writing any flow
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

# Quick sanity check — load iris to confirm sklearn is available
iris = load_iris(as_frame=True)
df = iris.frame
print(f"Iris dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head(3))


**What just happened?**

- We confirmed that `sklearn`, `pandas`, and `numpy` are importable in this Colab session.
- **`load_iris(as_frame=True)`** returns a `Bunch` whose `.frame` attribute is a ready-to-use DataFrame — no file downloads needed.
- The iris dataset has 150 rows and 5 columns (4 numeric features + 1 integer target), making it ideal for quick feature engineering demos.


## Step 2 · Writing a Multi-Step Feature Engineering Flow

A feature pipeline typically has these stages:

1. **Load** — ingest raw data
2. **Clean** — handle nulls, outliers, type issues
3. **Engineer** — create derived features
4. **Validate** — assert schema invariants before passing downstream

Splitting these into separate Metaflow steps means each is independently inspectable, retry-able, and re-usable. If feature engineering fails, you don't re-run the expensive data load.


In [ ]:
%%writefile feature_flow.py
from metaflow import FlowSpec, step
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris


class FeatureEngineeringFlow(FlowSpec):
    """Multi-step feature engineering pipeline for the Iris dataset."""

    @step
    def start(self):
        """Load raw data from sklearn — no external downloads."""
        iris = load_iris(as_frame=True)
        self.raw_df = iris.frame.copy()
        # Rename target to a readable name
        self.raw_df.rename(columns={'target': 'species'}, inplace=True)
        print(f"Loaded {len(self.raw_df)} rows")
        self.next(self.clean)

    @step
    def clean(self):
        """Drop nulls and clip extreme values (simulating real-world cleaning)."""
        df = self.raw_df.copy()

        # Inject some synthetic nulls so the cleaning step is non-trivial
        rng = np.random.default_rng(42)
        null_idx = rng.choice(df.index, size=5, replace=False)
        df.loc[null_idx, 'sepal length (cm)'] = np.nan

        before = len(df)
        df = df.dropna()  # drop rows with any null
        print(f"Dropped {before - len(df)} null rows — {len(df)} rows remain")

        # Clip numeric columns to [1st, 99th] percentile to remove outliers
        numeric_cols = df.select_dtypes(include='number').columns.tolist()
        numeric_cols = [c for c in numeric_cols if c != 'species']
        for col in numeric_cols:
            lo, hi = df[col].quantile([0.01, 0.99])
            df[col] = df[col].clip(lo, hi)

        self.clean_df = df
        self.next(self.engineer_features)

    @step
    def engineer_features(self):
        """Create derived features from the cleaned data."""
        df = self.clean_df.copy()

        # Ratio features: petal area proxy and sepal area proxy
        df['petal_area'] = df['petal length (cm)'] * df['petal width (cm)']
        df['sepal_area'] = df['sepal length (cm)'] * df['sepal width (cm)']

        # Interaction: petal-to-sepal length ratio
        df['petal_sepal_ratio'] = df['petal length (cm)'] / df['sepal length (cm)']

        # Log-transform skewed features (add small epsilon to avoid log(0))
        df['log_petal_area'] = np.log1p(df['petal_area'])

        print(f"Engineered features. Columns now: {list(df.columns)}")
        self.feature_df = df
        self.feature_columns = [c for c in df.columns if c != 'species']
        self.next(self.validate)

    @step
    def validate(self):
        """Assert schema invariants before passing features downstream."""
        df = self.feature_df

        # Invariant 1: no nulls in the feature matrix
        null_count = df[self.feature_columns].isnull().sum().sum()
        assert null_count == 0, f"Feature matrix has {null_count} nulls!"

        # Invariant 2: all feature values are finite
        inf_count = np.isinf(df[self.feature_columns].values).sum()
        assert inf_count == 0, f"Feature matrix has {inf_count} infinite values!"

        # Invariant 3: expected feature columns exist
        required = {'petal_area', 'sepal_area', 'petal_sepal_ratio', 'log_petal_area'}
        missing = required - set(df.columns)
        assert not missing, f"Missing engineered columns: {missing}"

        print("All validation checks passed!")
        self.next(self.end)

    @step
    def end(self):
        """Summarise the final feature matrix and store it as an artifact."""
        df = self.feature_df
        print(f"Final feature matrix: {df.shape}")
        print(df[self.feature_columns].describe().round(3))
        # self.feature_df is already persisted — downstream flows can access it
        # via: run = Flow('FeatureEngineeringFlow').latest_run
        #       run['end'].task.data.feature_df


if __name__ == '__main__':
    FeatureEngineeringFlow()


**What just happened?**

- `%%writefile` saved the flow to `feature_flow.py` on disk — we haven't run it yet.
- **Each step has a single responsibility**: load → clean → engineer → validate → end.
- `self.feature_df` set in `engineer_features` is automatically available in `validate` and `end` because Metaflow serialises all `self` attributes between steps.
- The `validate` step uses Python `assert` — if any invariant fails, the run stops and Metaflow marks it as failed, preserving all artifacts up to that point for debugging.


## Step 3 · Running the Flow and Inspecting Artifacts

Run a Metaflow flow with `python flow.py run`. After a successful run, you can access every artifact via the **Client API**: `Flow`, `Run`, `Step`, and `Task` objects form a hierarchy for browsing past runs.

```
Flow('FeatureEngineeringFlow')
  └── Run('FeatureEngineeringFlow/1')
       ├── Step('start') → Task → data.raw_df
       ├── Step('clean') → Task → data.clean_df
       └── Step('end')   → Task → data.feature_df
```


In [ ]:
# Run the flow — output is streamed to the notebook
!python feature_flow.py run


**What just happened?**

- Metaflow executed each `@step` method in sequence, printing progress and your `print()` calls.
- Every `self.X` assignment was serialised to a local datastore (typically `~/.metaflow/`).
- **The run ID** in the output (e.g., `FeatureEngineeringFlow/1`) lets you retrieve any artifact later via the Client API.
- If any assertion in `validate` had failed, Metaflow would have printed the traceback and marked the run as `failed` — the other steps' artifacts are still readable.


## Step 4 · Accessing Artifacts via the Client API

The Metaflow Client API lets you retrieve artifacts from past runs without re-running the flow. This is key for downstream flows — a model training flow can load the feature matrix produced by the feature pipeline.

| API object | How to get it |
|---|---|
| `Flow('Name')` | List all runs for a flow |
| `.latest_run` | Most recent run |
| `run['step_name']` | A Step within the run |
| `step.task` | The Task (single-instance steps have one task) |
| `task.data.artifact_name` | Read the artifact value |


In [ ]:
from metaflow import Flow

# Retrieve the latest run of our feature flow
run = Flow('FeatureEngineeringFlow').latest_run
print(f"Run ID: {run.id}  |  Successful: {run.successful}")

# Access the feature DataFrame artifact from the 'end' step
end_task = run['end'].task
feature_df = end_task.data.feature_df
feature_columns = end_task.data.feature_columns

print(f"\nFeature matrix shape: {feature_df.shape}")
print(f"Feature columns: {feature_columns}")
print(f"\nFirst 3 rows:")
print(feature_df[feature_columns].head(3).round(4))


**What just happened?**

- `Flow('FeatureEngineeringFlow').latest_run` fetched metadata from the local Metaflow datastore — no re-execution.
- **`task.data.feature_df`** deserialised the pandas DataFrame that was stored during the run. Any picklable Python object can be an artifact.
- In production, a downstream training flow would call this same API to load the latest feature matrix — decoupling the two pipelines while preserving full data lineage.


## Step 5 · Adding sklearn Transformers Inside Steps

You can use any Python library inside a Metaflow step. For sklearn transformers, **fit on training data and store the fitted transformer as an artifact** — this ensures your validation and inference splits use the same scaler parameters as training.

Best practice pattern:
```
fit_scaler step → self.scaler = fitted_scaler
transform_train step → self.X_train = scaler.transform(train)
transform_val step  → self.X_val  = scaler.transform(val)   # same scaler!
```


In [ ]:
%%writefile feature_flow_scaled.py
from metaflow import FlowSpec, step
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


class FeatureScalingFlow(FlowSpec):
    """Demonstrates fitting a scaler inside a step and reusing it."""

    @step
    def start(self):
        iris = load_iris(as_frame=True)
        df = iris.frame.rename(columns={'target': 'species'})

        # Engineer a couple of features
        df['petal_area'] = df['petal length (cm)'] * df['petal width (cm)']
        df['sepal_area'] = df['sepal length (cm)'] * df['sepal width (cm)']

        self.feature_cols = ['sepal length (cm)', 'sepal width (cm)',
                              'petal length (cm)', 'petal width (cm)',
                              'petal_area', 'sepal_area']

        X = df[self.feature_cols].values
        y = df['species'].values

        # Split before fitting the scaler — never fit on validation data
        self.X_train_raw, self.X_val_raw, self.y_train, self.y_val = \
            train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

        self.next(self.fit_scaler)

    @step
    def fit_scaler(self):
        """Fit StandardScaler on TRAINING data only, then transform both splits."""
        scaler = StandardScaler()
        # Fit only on training — prevents data leakage from validation set
        self.X_train = scaler.fit_transform(self.X_train_raw)
        self.X_val   = scaler.transform(self.X_val_raw)   # transform, not fit_transform
        self.scaler  = scaler  # persist so downstream inference can reuse it

        print(f"Train mean (should be ~0): {self.X_train.mean(axis=0).round(3)}")
        print(f"Train std  (should be ~1): {self.X_train.std(axis=0).round(3)}")
        self.next(self.end)

    @step
    def end(self):
        print(f"Train shape: {self.X_train.shape}")
        print(f"Val   shape: {self.X_val.shape}")
        print("Scaler params stored — downstream flows can load this artifact.")


if __name__ == '__main__':
    FeatureScalingFlow()


In [ ]:
!python feature_flow_scaled.py run


**What just happened?**

- The scaler was **fitted only on training data** — a critical ML best practice enforced structurally by having a dedicated `fit_scaler` step.
- `self.scaler` is stored as a Metaflow artifact — any downstream inference flow can load this exact fitted scaler and apply it to new data without re-fitting.
- **Training mean ≈ 0 and std ≈ 1** confirms StandardScaler worked correctly on the training split.
- The validation set transformation uses `transform` (not `fit_transform`) — the same mean/std from training is applied, preventing data leakage.


## Step 6 · Validating Feature Schema Invariants

A dedicated validation step acts as a **contract** between your feature pipeline and downstream consumers. Good invariants to check:

| Invariant | What to check |
|---|---|
| No nulls | `df.isnull().sum().sum() == 0` |
| No infinities | `np.isinf(df.values).sum() == 0` |
| Expected columns | `required_cols.issubset(df.columns)` |
| Value ranges | `df['petal_area'].min() >= 0` |
| Row count | `len(df) >= min_expected_rows` |
| No constant features | `df[feat_cols].std().min() > 0` |


In [ ]:
# Demonstrate a failing validation to see how Metaflow handles it
%%writefile validation_demo.py
from metaflow import FlowSpec, step
import pandas as pd
import numpy as np


class ValidationDemoFlow(FlowSpec):
    """Shows how a failed assertion stops the flow cleanly."""

    @step
    def start(self):
        # Deliberately create a DataFrame with a null
        self.df = pd.DataFrame({'a': [1.0, None, 3.0], 'b': [4.0, 5.0, 6.0]})
        print(f"Created DataFrame with {self.df.isnull().sum().sum()} null(s)")
        self.next(self.validate)

    @step
    def validate(self):
        null_count = self.df.isnull().sum().sum()
        # This assertion will FAIL — demonstrating the error behaviour
        assert null_count == 0, f"Validation failed: {null_count} null(s) found in feature matrix"
        self.next(self.end)

    @step
    def end(self):
        print("End (should not reach here)")


if __name__ == '__main__':
    ValidationDemoFlow()


In [ ]:
# This run WILL fail — that's intentional. Note how Metaflow reports the error.
!python validation_demo.py run 2>&1 | tail -20


**What just happened?**

- The `validate` step raised an `AssertionError` — Metaflow caught it and marked the run as **failed**.
- Despite the failure, the `start` step's artifacts (including `self.df`) are **still persisted** in the datastore — you can inspect them via the Client API to debug.
- **This is the key insight**: validation steps don't just print warnings, they hard-stop the pipeline. Downstream consumers (model training, serving) never see bad data.
- In production, a failed run triggers alerts (Slack, PagerDuty) via Metaflow's event system or your scheduler's failure hooks.


In [ ]:
# Challenge: Build a feature pipeline for the diabetes dataset
# Your solution here
#
# 1. Load sklearn.datasets.load_diabetes(as_frame=True)
# 2. Add a 'clean' step that drops rows where any feature is outside
#    3 standard deviations from the mean (outlier removal)
# 3. Add an 'engineer' step that creates:
#    - bmi_age_interaction = bmi * age
#    - bp_bmi_ratio = bp / (bmi + 1e-6)
# 4. Add a 'validate' step with at least 3 assertions
# 5. Run the flow and retrieve the feature_df artifact
#
# Scaffold:
# from metaflow import FlowSpec, step
# from sklearn.datasets import load_diabetes
# ...
#
# class DiabetesFeatureFlow(FlowSpec):
#     @step
#     def start(self):
#         ...
#         self.next(self.clean)
#     # add your steps here


---
## Day 10 key concepts recap

| Concept | What to remember |
|---|---|
| `FlowSpec` + `@step` | Every method decorated with `@step` is a DAG node; `self.next()` declares edges |
| Artifacts via `self` | Any `self.X = value` is automatically persisted between steps |
| Step granularity | Small steps = independent retry, inspection, and reuse |
| `fit` vs `transform` | Always fit scalers on training data only; store the fitted object as an artifact |
| Validation step | Use `assert` to hard-stop on bad data — downstream consumers are protected |
| Client API | `Flow(...).latest_run['step'].task.data.artifact` to read any past artifact |
| `%%writefile` + `!python` | The Colab/Jupyter pattern for running Metaflow flows interactively |

> **Tip:** Split feature engineering into small, testable steps — each step's output can be independently inspected and re-used.

---
## What's next
**Day 11** → Model Training and Experiment Tracking — build a training flow, parallelize hyperparameter search with `@foreach`, collect metrics in a join step, and visualize results with `@card`.

Mark Day 10 complete in your [tracker](../index.html).
